# XGBWW Random-10 Long-Run Alpha Tracking

This notebook extends the long-run checkpointed alpha-tracking workflow to a random sample of 10 datasets from **xgbwwdata**.

It combines:
- long-run chunked training + resume/checkpoint behavior
- WeightWatcher alpha tracking on `W1`, `W2`, `W7`, `W8`, `W9`, `W10`
- random multi-dataset registry workflow with robust error handling


In [ ]:
# Bootstrap installs (Colab-friendly)
%pip install -q xgboost weightwatcher scikit-learn pandas matplotlib seaborn scipy feather-format
!git clone --depth 1 https://github.com/CalculatedContent/xgboost2ww.git /tmp/xgboost2ww-src || true
!git clone --depth 1 https://github.com/CalculatedContent/xgbwwdata.git /tmp/xgbwwdata-src || true
%pip install -q /tmp/xgboost2ww-src
%pip install -q /tmp/xgbwwdata-src


In [ ]:
# Imports
import os
import json
import time
import random
import platform
import traceback
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
import weightwatcher as ww

from scipy import sparse
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

from xgboost2ww import convert
from xgbwwdata import Filters, load_dataset

try:
    from xgbwwdata import scan_datasets as xgbww_scan_datasets
except Exception:
    xgbww_scan_datasets = None

try:
    from xgbwwdata import list_datasets as xgbww_list_datasets
except Exception:
    xgbww_list_datasets = None


In [ ]:
# Runtime + global config
RANDOM_STATE = 42
DATASET_SAMPLE_SIZE = 10
TOTAL_ROUNDS = 1200
CHUNK_SIZE = 25
N_STEPS = TOTAL_ROUNDS // CHUNK_SIZE
CHECKPOINT_EVERY_STEPS = 1
RESUME_FROM_CHECKPOINT = True
FORCE_RESTART_ALL = False
FORCE_RESTART_DATASETS = []
SELECTED_DATASET_UIDS = []
DRY_RUN = False
MAX_DENSE_ELEMENTS = int(2e8)

if DRY_RUN:
    DATASET_SAMPLE_SIZE = 2
    TOTAL_ROUNDS = 50
    CHUNK_SIZE = 25
    N_STEPS = TOTAL_ROUNDS // CHUNK_SIZE

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
sns.set_theme(style='whitegrid')
print(f'N_STEPS={N_STEPS}, sample={DATASET_SAMPLE_SIZE}, dry_run={DRY_RUN}')


In [ ]:
# Colab / Drive config
in_colab = False
try:
    import google.colab  # noqa: F401
    from google.colab import drive
    in_colab = True
except Exception:
    drive = None

if in_colab:
    drive.mount('/content/drive')
    project_root = Path('/content/drive/MyDrive/xgboost2ww_runs/random10_longrun_alpha_tracking')
else:
    project_root = Path('./random10_longrun_alpha_tracking')

registry_dir = project_root / 'registry'
per_dataset_dir = project_root / 'per_dataset'
aggregate_dir = project_root / 'aggregate'
logs_dir = project_root / 'logs'
for p in [project_root, registry_dir, per_dataset_dir, aggregate_dir, logs_dir]:
    p.mkdir(parents=True, exist_ok=True)

errors_path = logs_dir / 'errors.csv'
skipped_path = aggregate_dir / 'skipped_datasets.csv'
print('Project root:', project_root.resolve())


In [ ]:
# Backend detection

def choose_xgb_runtime():
    cpu_threads = max(1, (os.cpu_count() or 1) - 1)
    is_apple_silicon = platform.system() == 'Darwin' and platform.machine() == 'arm64'
    if is_apple_silicon:
        out = {'backend_name': 'apple_silicon_cpu_hist', 'params_patch': {'tree_method': 'hist', 'nthread': cpu_threads}, 'is_gpu': False}
        print('[backend]', out)
        return out

    gpu_patch = {'tree_method': 'hist', 'device': 'cuda'}
    try:
        X_probe = np.array([[0.0], [1.0], [2.0], [3.0]], dtype=np.float32)
        y_probe = np.array([0, 0, 1, 1], dtype=np.float32)
        dprobe = xgb.DMatrix(X_probe, label=y_probe)
        xgb.train(params={'objective': 'binary:logistic', 'eval_metric': 'logloss', **gpu_patch}, dtrain=dprobe, num_boost_round=1, verbose_eval=False)
        out = {'backend_name': 'cuda_hist', 'params_patch': gpu_patch, 'is_gpu': True}
        print('[backend]', out)
        return out
    except Exception:
        out = {'backend_name': 'cpu_hist', 'params_patch': {'tree_method': 'hist', 'nthread': cpu_threads}, 'is_gpu': False}
        print('[backend]', out)
        return out

BACKEND_INFO = choose_xgb_runtime()


In [ ]:
# xgbwwdata scan / registry build

def build_registry_df(filters: Filters) -> pd.DataFrame:
    if xgbww_scan_datasets is not None:
        reg = xgbww_scan_datasets(filters=filters, preprocess=True, smoke_train=True)
        return pd.DataFrame(reg)
    if xgbww_list_datasets is not None:
        reg = xgbww_list_datasets(filters=filters, preprocess=True, smoke_train=True)
        return pd.DataFrame(reg)
    try:
        from xgbwwdata import get_registry
        return pd.DataFrame(get_registry(filters=filters, preprocess=True, smoke_train=True))
    except Exception as e:
        raise RuntimeError('No compatible xgbwwdata scan/list API found in this environment.') from e

filters = Filters(min_rows=200, max_rows=60000, max_features=50000, max_dense_elements=MAX_DENSE_ELEMENTS, preprocess=True)
full_registry_csv = registry_dir / 'full_registry.csv'
full_registry_feather = registry_dir / 'full_registry.feather'

if RESUME_FROM_CHECKPOINT and full_registry_csv.exists() and not FORCE_RESTART_ALL:
    full_registry_df = pd.read_csv(full_registry_csv)
    print(f'Loaded cached full registry: {len(full_registry_df)} rows')
else:
    full_registry_df = build_registry_df(filters)
    full_registry_df.to_csv(full_registry_csv, index=False)
    try:
        full_registry_df.reset_index(drop=True).to_feather(full_registry_feather)
    except Exception as e:
        print('[WARN] Feather save failed for full registry:', e)
    print(f'Scanned full registry: {len(full_registry_df)} rows')

registry_df = full_registry_df.copy()
if 'task' in registry_df.columns:
    registry_df = registry_df[registry_df['task'].astype(str).str.contains('class', case=False, na=False)]
if 'n_classes' in registry_df.columns:
    registry_df = registry_df[(registry_df['n_classes'].fillna(0) >= 2) & (registry_df['n_classes'].fillna(0) <= 20)]
registry_df = registry_df.reset_index(drop=True)
print('Filtered registry rows:', len(registry_df))


In [ ]:
# Random sample of 10 datasets
random10_csv = registry_dir / 'random10_registry.csv'
random10_feather = registry_dir / 'random10_registry.feather'

if SELECTED_DATASET_UIDS:
    sampled_registry = registry_df[registry_df['dataset_uid'].isin(SELECTED_DATASET_UIDS)].copy()
    print(f'Using explicit SELECTED_DATASET_UIDS ({len(sampled_registry)} rows).')
elif RESUME_FROM_CHECKPOINT and random10_csv.exists() and not FORCE_RESTART_ALL:
    sampled_registry = pd.read_csv(random10_csv)
    print(f'Loaded prior sampled registry: {len(sampled_registry)} rows')
else:
    n_take = min(DATASET_SAMPLE_SIZE, len(registry_df))
    sampled_registry = registry_df.sample(n=n_take, random_state=RANDOM_STATE).copy()
    sampled_registry.to_csv(random10_csv, index=False)
    try:
        sampled_registry.reset_index(drop=True).to_feather(random10_feather)
    except Exception as e:
        print('[WARN] Feather save failed for sampled registry:', e)
    print(f'Sampled {len(sampled_registry)} datasets')

sampled_registry = sampled_registry.reset_index(drop=True)
display(sampled_registry.head(20))


In [ ]:
# Helper functions
W_LIST = ['W1', 'W2', 'W7', 'W8', 'W9', 'W10']

def make_safe_slug(text):
    text = str(text)
    out = ''.join(ch.lower() if ch.isalnum() else '_' for ch in text)
    while '__' in out:
        out = out.replace('__', '_')
    return out.strip('_')[:120]

def log_error(error_rows, dataset_uid, dataset_slug, stage, exc):
    error_rows.append({'dataset_uid': dataset_uid, 'dataset_slug': dataset_slug, 'stage': stage, 'error_type': type(exc).__name__, 'error_message': str(exc), 'traceback': traceback.format_exc(), 'timestamp': pd.Timestamp.utcnow().isoformat()})

def save_error_log(error_rows, errors_path):
    if error_rows:
        pd.DataFrame(error_rows).to_csv(errors_path, index=False)

def make_dataset_paths(dataset_slug):
    base = per_dataset_dir / dataset_slug
    out = {'base': base, 'results': base / 'results', 'models': base / 'models', 'plots': base / 'plots'}
    for p in out.values():
        p.mkdir(parents=True, exist_ok=True)
    return out

def detect_task_type(y):
    y_np = np.asarray(y)
    n_classes = int(len(np.unique(y_np)))
    if n_classes < 2:
        return 'invalid', n_classes, y_np
    return ('binary' if n_classes == 2 else 'multiclass'), n_classes, y_np

def build_params_for_dataset(base_backend_patch, n_classes):
    params = {'max_depth': 4, 'eta': 0.05, 'subsample': 0.9, 'colsample_bytree': 0.8, 'min_child_weight': 5, 'reg_lambda': 5.0, 'reg_alpha': 0.5, 'gamma': 1.0, 'seed': RANDOM_STATE, **base_backend_patch}
    if n_classes == 2:
        params.update({'objective': 'binary:logistic', 'eval_metric': 'logloss'})
    else:
        params.update({'objective': 'multi:softprob', 'eval_metric': 'mlogloss', 'num_class': int(n_classes)})
    return params

def prepare_train_test_data(X, y):
    stratify = y if len(np.unique(y)) > 1 else None
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=stratify)
    scaler = StandardScaler(with_mean=False) if sparse.issparse(X_train) else StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train).astype(np.float32)
    X_test_scaled = scaler.transform(X_test).astype(np.float32)
    y_train_np = np.asarray(y_train).astype(np.int32).reshape(-1)
    y_test_np = np.asarray(y_test).astype(np.int32).reshape(-1)
    return X_train_scaled, X_test_scaled, y_train_np, y_test_np

def densify_for_convert_if_safe(X_train_scaled, max_dense_elements):
    if sparse.issparse(X_train_scaled):
        n_elements = int(X_train_scaled.shape[0]) * int(X_train_scaled.shape[1])
        if n_elements > max_dense_elements:
            raise MemoryError(f'Refusing to densify convert input: shape={X_train_scaled.shape}, elements={n_elements:,}')
        return X_train_scaled.toarray().astype(np.float32)
    return np.asarray(X_train_scaled).astype(np.float32)

def build_layer_for_W(bst, W_name, current_round, X_train_for_convert, y_train_np, params):
    multiclass_mode = 'softprob' if int(params.get('num_class', 2)) > 2 else 'error'
    return convert(model=bst, data=X_train_for_convert, labels=y_train_np, W=W_name, return_type='torch', nfolds=5, t_points=min(current_round, 160), random_state=RANDOM_STATE, train_params=params, num_boost_round=current_round, multiclass=multiclass_mode)

def compute_alpha_from_layer(layer):
    watcher = ww.WeightWatcher(model=layer)
    df = watcher.analyze(randomize=True, detX=True)
    return float(df['alpha'].iloc[0])

def save_dataset_checkpoint(rows, metrics_csv_path, metrics_feather_path, bst, current_round, models_dir, metadata):
    df = pd.DataFrame(rows)
    df.to_csv(metrics_csv_path, index=False)
    try:
        df.reset_index(drop=True).to_feather(metrics_feather_path)
    except Exception as e:
        print('[WARN] Feather save failed:', e)
    bst.save_model(models_dir / f'model_round_{current_round:04d}.json')
    bst.save_model(models_dir / 'model_latest.json')
    metadata = dict(metadata)
    metadata['last_round'] = int(current_round)
    metadata['updated_at'] = pd.Timestamp.utcnow().isoformat()
    (models_dir / 'metadata.json').write_text(json.dumps(metadata, indent=2))

def load_dataset_checkpoint(metrics_csv_path, latest_model_path, chunk_size, n_steps):
    if not (metrics_csv_path.exists() and latest_model_path.exists()):
        return None, 1, []
    prior_df = pd.read_csv(metrics_csv_path)
    if prior_df.empty:
        return None, 1, []
    max_round = int(prior_df['boosting_round'].max())
    if max_round % chunk_size != 0:
        raise ValueError(f'Checkpoint round {max_round} incompatible with CHUNK_SIZE={chunk_size}')
    completed_steps = max_round // chunk_size
    if completed_steps > n_steps:
        raise ValueError(f'Checkpoint completed_steps={completed_steps} > N_STEPS={n_steps}')
    bst = xgb.Booster()
    bst.load_model(latest_model_path)
    return bst, completed_steps + 1, prior_df.to_dict('records')

def plot_dataset_dynamics(df_dataset, dataset_slug, plots_dir):
    if df_dataset.empty:
        return None, None
    long_alpha = df_dataset.melt(id_vars=['boosting_round', 'test_accuracy'], value_vars=[f'alpha_{w}' for w in W_LIST], var_name='alpha_name', value_name='alpha_value')
    fig, axes = plt.subplots(1, 3, figsize=(20, 5))
    axes[0].plot(df_dataset['boosting_round'], df_dataset['test_accuracy'], marker='o')
    axes[0].set_title(f'{dataset_slug}: test accuracy vs round')
    sns.lineplot(data=long_alpha, x='boosting_round', y='alpha_value', hue='alpha_name', ax=axes[1])
    axes[1].set_title('alpha vs round')
    sns.scatterplot(data=long_alpha, x='alpha_value', y='test_accuracy', hue='alpha_name', ax=axes[2])
    axes[2].set_title('accuracy vs alpha')
    fig.tight_layout()
    p1 = plots_dir / f'{dataset_slug}_dynamics.png'
    fig.savefig(p1, dpi=140)
    plt.close(fig)
    fig2, ax2 = plt.subplots(figsize=(7, 5))
    for w in W_LIST:
        col = f'alpha_{w}'
        ax2.scatter(df_dataset[col], df_dataset['test_accuracy'], alpha=0.6, label=w)
    ax2.set_title(f'{dataset_slug}: accuracy vs alpha (all W)')
    ax2.legend(ncol=2)
    fig2.tight_layout()
    p2 = plots_dir / f'{dataset_slug}_accuracy_vs_alpha.png'
    fig2.savefig(p2, dpi=140)
    plt.close(fig2)
    return p1, p2

def summarize_dataset_results(df_dataset):
    if df_dataset.empty:
        return {}
    best_row = df_dataset.loc[df_dataset['test_accuracy'].idxmax()].to_dict()
    corr = {}
    for w in W_LIST:
        c = df_dataset['test_accuracy'].corr(df_dataset[f'alpha_{w}'])
        corr[f'corr_acc_alpha_{w}'] = float(c) if pd.notnull(c) else np.nan
    return {'best_row': best_row, **corr}


In [ ]:
# Per-dataset training loop
all_results, skipped_rows, error_rows = [], [], []

for i, row in sampled_registry.iterrows():
    dataset_uid = row.get('dataset_uid', f'row_{i}')
    source = row.get('source', 'unknown')
    dataset_slug = make_safe_slug(f"{dataset_uid}_{source}")
    print(f"\n[{i+1}/{len(sampled_registry)}] dataset_uid={dataset_uid} source={source}")
    force_restart_this = FORCE_RESTART_ALL or (dataset_slug in FORCE_RESTART_DATASETS) or (dataset_uid in FORCE_RESTART_DATASETS)

    paths = make_dataset_paths(dataset_slug)
    metrics_csv = paths['results'] / f'{dataset_slug}_metrics.csv'
    metrics_feather = paths['results'] / f'{dataset_slug}_metrics.feather'
    latest_model_path = paths['models'] / 'model_latest.json'

    try:
        X, y, meta = load_dataset(dataset_uid, filters=filters)
    except Exception as e:
        log_error(error_rows, dataset_uid, dataset_slug, 'dataset_load', e)
        skipped_rows.append({'dataset_uid': dataset_uid, 'dataset_slug': dataset_slug, 'reason': 'dataset_load_failed'})
        save_error_log(error_rows, errors_path)
        continue

    try:
        task_type, n_classes, y_np = detect_task_type(y)
        if task_type == 'invalid' or n_classes > 20:
            raise ValueError(f'Invalid class count: n_classes={n_classes}')
        X_train_scaled, X_test_scaled, y_train_np, y_test_np = prepare_train_test_data(X, y_np)
        X_train_for_convert = densify_for_convert_if_safe(X_train_scaled, MAX_DENSE_ELEMENTS)
    except Exception as e:
        log_error(error_rows, dataset_uid, dataset_slug, 'split_preprocess', e)
        skipped_rows.append({'dataset_uid': dataset_uid, 'dataset_slug': dataset_slug, 'reason': 'split_preprocess_failed'})
        save_error_log(error_rows, errors_path)
        continue

    dtrain, dtest = xgb.DMatrix(X_train_scaled, label=y_train_np), xgb.DMatrix(X_test_scaled, label=y_test_np)
    params = build_params_for_dataset(BACKEND_INFO['params_patch'], n_classes)

    bst, rows, start_step = None, [], 1
    if (not force_restart_this) and RESUME_FROM_CHECKPOINT:
        try:
            bst, start_step, rows = load_dataset_checkpoint(metrics_csv, latest_model_path, CHUNK_SIZE, N_STEPS)
            if bst is not None:
                print(f'[RESUME] step={start_step}/{N_STEPS}')
        except Exception as e:
            log_error(error_rows, dataset_uid, dataset_slug, 'checkpoint_load', e)
            bst, rows, start_step = None, [], 1

    dataset_t0 = time.time()
    dataset_failed = False
    for step in range(start_step, N_STEPS + 1):
        current_round = step * CHUNK_SIZE
        step_t0 = time.time()
        try:
            bst = xgb.train(params=params, dtrain=dtrain, num_boost_round=CHUNK_SIZE, xgb_model=bst, verbose_eval=False)
            y_prob = bst.predict(dtest)
            y_pred = (y_prob >= 0.5).astype(np.int32) if n_classes == 2 else np.argmax(y_prob, axis=1).astype(np.int32)
            test_accuracy = float(accuracy_score(y_test_np, y_pred))
        except Exception as e:
            log_error(error_rows, dataset_uid, dataset_slug, 'train_or_predict', e)
            dataset_failed = True
            break

        alpha_values, alpha_failures = {}, 0
        for w_name in W_LIST:
            try:
                layer = build_layer_for_W(bst, w_name, current_round, X_train_for_convert, y_train_np, params)
                alpha_values[f'alpha_{w_name}'] = compute_alpha_from_layer(layer)
            except Exception as e:
                alpha_values[f'alpha_{w_name}'] = np.nan
                alpha_failures += 1
                log_error(error_rows, dataset_uid, dataset_slug, f'alpha_{w_name}', e)

        if alpha_failures == len(W_LIST):
            dataset_failed = True
            log_error(error_rows, dataset_uid, dataset_slug, 'alpha_all_failed', RuntimeError('All W matrices failed this round'))
            break

        rows.append({'dataset_uid': dataset_uid, 'dataset_slug': dataset_slug, 'source': source, 'n_samples': int(np.asarray(y_np).shape[0]), 'n_features': int(X_train_scaled.shape[1]), 'n_classes': int(n_classes), 'boosting_round': int(current_round), 'test_accuracy': test_accuracy, **alpha_values, 'backend': BACKEND_INFO.get('backend_name'), 'tree_method': params.get('tree_method'), 'device': params.get('device', 'cpu'), 'elapsed_seconds_round': float(time.time() - step_t0), 'elapsed_seconds_total': float(time.time() - dataset_t0)})

        if step % CHECKPOINT_EVERY_STEPS == 0:
            try:
                save_dataset_checkpoint(rows, metrics_csv, metrics_feather, bst, current_round, paths['models'], {'dataset_uid': dataset_uid, 'dataset_slug': dataset_slug, 'source': source, 'params': params, 'backend_info': BACKEND_INFO})
            except Exception as e:
                log_error(error_rows, dataset_uid, dataset_slug, 'checkpoint_save', e)

        print(f"  step={step:03d}/{N_STEPS} round={current_round:04d} acc={test_accuracy:.4f}")

    df_dataset = pd.DataFrame(rows)
    if not df_dataset.empty:
        all_results.append(df_dataset)
        try:
            plot_dataset_dynamics(df_dataset, dataset_slug, paths['plots'])
        except Exception as e:
            log_error(error_rows, dataset_uid, dataset_slug, 'plotting', e)
    if dataset_failed and df_dataset.empty:
        skipped_rows.append({'dataset_uid': dataset_uid, 'dataset_slug': dataset_slug, 'reason': 'dataset_failed'})

    save_error_log(error_rows, errors_path)

print('Done processing sampled datasets.')


In [ ]:
# Aggregate summary / plots
all_df = pd.concat(all_results, ignore_index=True) if all_results else pd.DataFrame()

all_csv = aggregate_dir / 'all_datasets_longrun_metrics.csv'
all_feather = aggregate_dir / 'all_datasets_longrun_metrics.feather'
best_csv = aggregate_dir / 'best_rows_per_dataset.csv'
corr_csv = aggregate_dir / 'correlation_summary.csv'
run_summary_json = aggregate_dir / 'run_summary.json'

if not all_df.empty:
    all_df.to_csv(all_csv, index=False)
    try:
        all_df.reset_index(drop=True).to_feather(all_feather)
    except Exception as e:
        print('[WARN] Feather save failed for aggregate metrics:', e)

    best_df = all_df.loc[all_df.groupby('dataset_uid')['test_accuracy'].idxmax()].sort_values('test_accuracy', ascending=False)
    best_df.to_csv(best_csv, index=False)

    corr_rows = []
    for dataset_uid, g in all_df.groupby('dataset_uid'):
        row = {'dataset_uid': dataset_uid, 'dataset_slug': g['dataset_slug'].iloc[0]}
        for w in W_LIST:
            c = g['test_accuracy'].corr(g[f'alpha_{w}'])
            row[f'corr_test_accuracy_alpha_{w}'] = float(c) if pd.notnull(c) else np.nan
        corr_rows.append(row)
    corr_df = pd.DataFrame(corr_rows)
    corr_df.to_csv(corr_csv, index=False)

    rank_rows = []
    for w in W_LIST:
        cvals = corr_df[f'corr_test_accuracy_alpha_{w}'].dropna()
        rank_rows.append({'W': w, 'avg_abs_corr': float(np.abs(cvals).mean()) if len(cvals) else np.nan, 'share_lower_alpha_better': float((cvals < 0).mean()) if len(cvals) else np.nan})
    rank_df = pd.DataFrame(rank_rows).sort_values('avg_abs_corr', ascending=False)

    run_summary = {'project_root': str(project_root), 'n_sampled_datasets': int(len(sampled_registry)), 'n_completed_datasets': int(all_df['dataset_uid'].nunique()), 'n_rows_total': int(len(all_df)), 'backend': BACKEND_INFO, 'timestamp_utc': pd.Timestamp.utcnow().isoformat()}
    run_summary_json.write_text(json.dumps(run_summary, indent=2))

    if skipped_rows:
        pd.DataFrame(skipped_rows).to_csv(skipped_path, index=False)

    display(sampled_registry)
    display(pd.DataFrame(skipped_rows) if skipped_rows else pd.DataFrame(columns=['dataset_uid', 'dataset_slug', 'reason']))
    display(best_df)
    display(all_df.sort_values('test_accuracy', ascending=False).head(20))
    display(corr_df)
    display(rank_df)
else:
    print('No completed dataset results to aggregate.')

try:
    import shutil
    archive_path = shutil.make_archive(str(aggregate_dir / 'aggregate_exports'), 'zip', root_dir=aggregate_dir)
    print('Aggregate zip export:', archive_path)
except Exception as e:
    print('[WARN] zip export skipped:', e)


## Final interpretation

This notebook is the **random-10 multi-dataset extension** of `AdultIncome_LongRun_Alpha_Tracking.ipynb`.

It preserves:
- long-run chunked XGBoost training with checkpoint/resume
- per-round alpha tracking from WeightWatcher for `W1`, `W2`, `W7`, `W8`, `W9`, `W10`
- robust save/restart behavior across Colab Drive and local Jupyter

The main purpose is to test whether HTSR-style alpha behavior repeats across heterogeneous datasets. Because datasets differ in source, size, feature geometry, and class structure, aggregate trends should be interpreted carefully and validated with follow-up experiments.
